# Isaac Sim Jupyter Notebook Tutorial

This notebook loads the IAI apartment and a Stretch robot, attaches a camera,
and drives the robot over ROS 2 (`cmd_vel` / `joint_states` / camera image).

> Select the jupyter kernel `Isaac Sim Python 3.11` if you are running in VScode.
>
> "Control + Enter" to execute the selected code cell.

## Start the GPU monitor and virtual desktop

Make sure the GPU has at least 3000M free memory.

If the virtual desktop does not pop up, manually click the "Open Desktop in new Tab".

In [ ]:
from IPython import get_ipython
in_notebook = get_ipython().__class__.__name__ == "ZMQInteractiveShell"

import os
import shutil
from utils import *
from pathlib import Path

# Current script directory
try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path(os.getcwd())

# Copy the precompiled kit cache if it is not present yet
target_dir = "/isaac-sim/kit/cache"
source_dir = "/mnt/isaacsim-cache/cache"
if os.path.isdir(source_dir) and not os.path.isdir(target_dir):
    shutil.copytree(source_dir, target_dir)

# only runs in Jupyter Notebook
if in_notebook:
    from gpu_monitor import GPUMonitor
    gpu_monitor = GPUMonitor()
    display_desktop()

## Start SimulationApp

The application window is frozen and non-interactive, which is normal.

<div style="color:red">This will take some time, so give it a minute and wait for it to say <b>"SimulationApp Ready!"</b> before you go to next step.</div>

In [ ]:
from isaacsim import SimulationApp
from IPython.display import clear_output
import sys

original_stdout = sys.stdout
original_stderr = sys.stderr

simulation_app = SimulationApp({
    "headless": False,
    "hide_ui": False,
    "width": 1280,
    "height": 960,
    "renderer": "RaytracedLighting",
    "display_options": 3286,  # show the default grid
})

# Fix the issue where notebook output is being hijacked by Isaac Sim.
sys.stdout = original_stdout
sys.stderr = original_stderr
clear_output(wait=True)
print('SimulationApp Ready!')

## Create the simulation environment

Physics updates at 200 Hz and rendering at 25 Hz. We load the ground, the
apartment, a few lights, and point the viewport camera at the scene.

In [ ]:
import numpy as np
from isaacsim.core.api import World
from isaacsim.core.utils.prims import define_prim, create_prim
from isaacsim.core.utils import viewports
from isaacsim.core.utils.extensions import enable_extension

enable_extension("isaacsim.ros2.bridge")

my_world = World(stage_units_in_meters=1.0, physics_dt=1 / 200, rendering_dt=8 / 200)
my_world.reset()

# Ground
define_prim("/World/Ground", "Xform").GetReferences().AddReference(
    f"{BASE_DIR}/../usd/Grid/default_environment.usd"
)

# Apartment
create_prim(
    usd_path=f"{BASE_DIR}/../usd/apartment/apartmentICRA.usda",
    prim_path="/World/Apartment",
    position=np.array([-6, 5, 0.0701]),
)

# Lights so the raytraced scene is not black
for i in range(1, 4):
    create_prim(
        prim_path=f"/World/Ground/Light_{i}",
        prim_type="SphereLight",
        attributes={"inputs:intensity": 10000},
        position=(-4 * i, 0, 2),
    )

viewports.set_camera_view(eye=np.array([-7, -2, 2]), target=np.array([-1, 1, 1]))

for _ in range(30):
    my_world.step(render=True)

## Spawn the Stretch robot

In [ ]:
from isaacsim.core.prims import Articulation

create_prim(
    usd_path=f"{BASE_DIR}/../usd/stretch/stretch.usd",
    prim_path="/World/stretch",
    position=np.array([-1.5, 0, 0.05]),
    orientation=np.array([0, 0, 0, 1]),
)

stretch = Articulation(prim_paths_expr="/World/stretch", name="stretch")
my_world.reset()

for _ in range(10):
    my_world.step(render=True)

## Add a head camera

In [ ]:
import omni
from pxr import UsdGeom
from isaacsim.sensors.camera import Camera
import isaacsim.core.utils.numpy.rotations as rot_utils

head_cam_prim = "/World/stretch/link_head_tilt/camera_bottom_screw_frame/camera_link/camera_color_frame/camera_color_optical_frame"

UsdGeom.Camera.Define(omni.usd.get_context().get_stage(), head_cam_prim)

head_cam = Camera(
    prim_path=head_cam_prim,
    frequency=30,
    resolution=(640, 360),
    orientation=rot_utils.euler_angles_to_quats(np.array([-90, 0, 0]), degrees=True),
)
head_cam.initialize()
head_cam.set_focal_length(1.5)
head_cam.set_clipping_range(near_distance=0.01, far_distance=20)

for _ in range(20):
    my_world.step(render=True)

## ROS 2 bridge node

A single node for the Stretch robot:
- subscribes to `/stretch/cmd_vel` and drives the wheels,
- subscribes to `/stretch/joint_command` and sets joint position targets,
- publishes `/stretch/joint_states` and `/head_camera/image_raw`,
- broadcasts TF for the full link tree (`map` → `base_link` → every link).

In [ ]:
import rclpy
from rclpy.node import Node
from geometry_msgs.msg import Twist, TransformStamped
from sensor_msgs.msg import JointState, Image
from tf2_ros import TransformBroadcaster

if not rclpy.ok():
    rclpy.init(args=None)


# --- tiny quaternion helpers (scalar-last x, y, z, w), no external deps ---
def _qconj(q):
    return np.array([-q[0], -q[1], -q[2], q[3]])


def _qmul(a, b):
    ax, ay, az, aw = a
    bx, by, bz, bw = b
    return np.array([
        aw * bx + ax * bw + ay * bz - az * by,
        aw * by - ax * bz + ay * bw + az * bx,
        aw * bz + ax * by - ay * bx + az * bw,
        aw * bw - ax * bx - ay * by - az * bz,
    ])


def _qrot(q, v):
    u = q[:3]
    s = q[3]
    return 2 * np.dot(u, v) * u + (s * s - np.dot(u, u)) * v + 2 * s * np.cross(u, v)


def _as_np(x):
    if hasattr(x, "numpy"):  # warp array or torch CPU tensor
        try:
            return x.numpy()
        except Exception:
            return x.detach().cpu().numpy()
    return np.asarray(x)


class StretchROS(Node):
    def __init__(self, robot, head_cam, prefix="stretch"):
        super().__init__(f"{prefix}_ros")
        self.robot = robot
        self.head_cam = head_cam

        # Differential base geometry
        self.wheel_base = 0.3407
        self.wheel_radius = 0.0125
        self.factor = 0.2
        names = np.array(robot.dof_names)
        self.left_wheel = np.where(np.char.find(names, "left_wheel") >= 0)[0]
        self.right_wheel = np.where(np.char.find(names, "right_wheel") >= 0)[0]

        # TF: link names and the base link index
        self.body_names = list(robot.body_names)
        self.base_idx = next(
            (i for i, n in enumerate(self.body_names) if "base_link" in n), 0
        )

        self.create_subscription(Twist, f"/{prefix}/cmd_vel", self.cmd_vel_cb, 10)
        self.create_subscription(JointState, f"/{prefix}/joint_command", self.joint_cmd_cb, 10)
        self.pub_js = self.create_publisher(JointState, f"/{prefix}/joint_states", 10)
        self.pub_img = self.create_publisher(Image, "/head_camera/image_raw", 10)
        self.tf_broadcaster = TransformBroadcaster(self)

    def cmd_vel_cb(self, msg):
        v, w = msg.linear.x, msg.angular.z
        v_left = (v - w * self.wheel_base / 2) / self.wheel_radius
        v_right = (v + w * self.wheel_base / 2) / self.wheel_radius
        vel = np.zeros(self.robot.num_dof)
        vel[self.left_wheel] = v_left * self.factor
        vel[self.right_wheel] = v_right * self.factor
        self.robot.set_joint_velocities([vel])

    def joint_cmd_cb(self, msg):
        target = self.robot.get_joint_positions()[0]
        for name, pos in zip(msg.name, msg.position):
            target[self.robot.get_dof_index(name)] = pos
        self.robot.set_joint_position_targets([target])

    def publish_joint_states(self):
        msg = JointState()
        msg.header.stamp = self.get_clock().now().to_msg()
        msg.name = list(self.robot.dof_names)
        msg.position = self.robot.get_joint_positions()[0].tolist()
        self.pub_js.publish(msg)

    def publish_camera(self):
        rgb = self.head_cam.get_rgba()[:, :, :3]
        msg = Image()
        msg.height = rgb.shape[0]
        msg.width = rgb.shape[1]
        msg.encoding = "rgb8"
        msg.step = rgb.shape[1] * 3
        msg.data = np.ascontiguousarray(rgb, dtype=np.uint8).tobytes()
        self.pub_img.publish(msg)

    def publish_tf(self):
        # World pose of every link: (num_links, 7) = x, y, z, qx, qy, qz, qw.
        # _physics_view is private but the only non-OmniGraph way to read all link poses.
        lt = _as_np(self.robot._physics_view.get_link_transforms()).reshape(-1, 7)
        pb, qb = lt[self.base_idx, :3], lt[self.base_idx, 3:7]
        qb_inv = _qconj(qb)
        now = self.get_clock().now().to_msg()

        tfs = [self._make_tf(now, "map", "base_link", pb, qb)]  # map -> base_link
        for i, name in enumerate(self.body_names):  # base_link -> every other link
            if i == self.base_idx:
                continue
            p_rel = _qrot(qb_inv, lt[i, :3] - pb)
            q_rel = _qmul(qb_inv, lt[i, 3:7])
            tfs.append(self._make_tf(now, "base_link", name, p_rel, q_rel))

        self.tf_broadcaster.sendTransform(tfs)

    @staticmethod
    def _make_tf(stamp, parent, child, p, q):
        t = TransformStamped()
        t.header.stamp = stamp
        t.header.frame_id = parent
        t.child_frame_id = child
        t.transform.translation.x = float(p[0])
        t.transform.translation.y = float(p[1])
        t.transform.translation.z = float(p[2])
        t.transform.rotation.x = float(q[0])
        t.transform.rotation.y = float(q[1])
        t.transform.rotation.z = float(q[2])
        t.transform.rotation.w = float(q[3])
        return t


stretch_node = StretchROS(stretch, head_cam, prefix="stretch")

## Open `rviz2`

In [ ]:
import subprocess

os.environ.pop("PYTHONPATH", None)
launch = '''
source /opt/ros/jazzy/setup.bash
rviz2 -d ./camera.rviz
'''
subprocess.Popen(["bash", "-c", launch],
                 stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

## Run the simulation loop

Drag the `rqt_robot_steering` sliders (or publish to `/stretch/cmd_vel`) to drive
the robot. Rendering and the camera publish every 3rd step to keep physics fast.

In [ ]:
from tqdm import tqdm

steps = 3000
bar_format = "{l_bar}{bar}| {n_fmt}/{total_fmt} steps] {elapsed_s:.2f}s"

for frame in tqdm(range(steps), desc="ROS Spin", ncols=60, bar_format=bar_format, file=sys.stdout):
    my_world.step(render=True)
    rclpy.spin_once(stretch_node, timeout_sec=0.0)
    stretch_node.publish_joint_states()
    stretch_node.publish_tf()
    stretch_node.publish_camera()

## Shutdown

In [ ]:
stretch_node.destroy_node()
rclpy.shutdown()
simulation_app.close()

## Convert notebook to Python script

```
jupyter nbconvert --to python apartment.ipynb
```